# E($n$)-Equivariant Spherical Decision Surfaces: Downstream Check

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import sys
SEED = 42

import numpy as np
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim
from torch import Tensor

import sklearn
from sklearn.utils import shuffle

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)

device = cpu


In [3]:
print(f'Versions: python {sys.version}, torch {torch.__version__}, numpy {np.__version__}, sklearn {sklearn.__version__}')

Versions: python 3.10.12 (main, Jul  5 2023, 15:02:25) [Clang 14.0.6 ], torch 2.1.0, numpy 1.25.0, sklearn 1.3.0


In [4]:
# ----------------------------
# Dataset
# ----------------------------
def random_on_matrix(n=3, reflection_prob=0.5):
    A = np.random.randn(n, n)
    Q, _ = np.linalg.qr(A)
    if np.linalg.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]
    if np.random.rand() < reflection_prob:
        Q[:, 0] = -Q[:, 0]
    return Q

def random_on_matrix_batch(B=1, n=3, reflection_prob=0.5):
    A = np.random.randn(B, n, n)
    Q, R = np.linalg.qr(A)
    signs = np.sign(np.diagonal(R, axis1=1, axis2=2))
    Q *= signs[:, None, :]

    dets = np.linalg.det(Q)
    neg = dets < 0
    Q[neg, :, 0] *= -1

    reflect = np.random.rand(B) < reflection_prob
    Q[reflect, :, 0] *= -1
    return Q

def get_tetris_data(total_size=10000, train_size=1000, shuffle_data=False,
                    distortion=None, only_canonical=False, only_label_names=False,
                    reflection_prob_train=0.5, reflection_prob_test=0.5):
    assert train_size < total_size

    label_names = ['chiral_shape_1', 'square', 'line', 'corner', 'L', 'T', 'zigzag']
    if only_label_names:
        return label_names

    tetris = [
        [(0, 0, 0), (0, 0, 1), (1, 0, 0), (1, 1, 0)],  # chiral_shape_1
        [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0)],  # square
        [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3)],  # line
        [(0, 0, 0), (0, 0, 1), (0, 1, 0), (1, 0, 0)],  # corner
        [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 0)],  # L
        [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 1)],  # T
        [(0, 0, 0), (1, 0, 0), (1, 1, 0), (2, 1, 0)]   # zigzag
    ]

    dataset = [np.array(points_, dtype=np.float32) for points_ in tetris]
    Xcanon = np.array(dataset, dtype=np.float32)
    Ycanon = np.arange(len(dataset), dtype=np.int64)

    if only_canonical:
        return torch.from_numpy(Xcanon).float(), torch.from_numpy(Ycanon).long()

    Xall, Yall = [], []
    j = 0
    per_class = total_size // len(tetris)

    for _ in range(per_class):
        for label, shape in enumerate(dataset):
            if j < train_size:
                O = random_on_matrix(n=3, reflection_prob=reflection_prob_train)
            else:
                O = random_on_matrix(n=3, reflection_prob=reflection_prob_test)

            transformed = shape @ O
            t = np.random.uniform(low=-3.0, high=3.0, size=(1, 3)).astype(np.float32)
            transformed = transformed + t

            if distortion:
                transformed += np.random.uniform(low=-distortion, high=distortion, size=(4, 3)).astype(np.float32)

            Xall.append(transformed.astype(np.float32))
            Yall.append(label)
            j += 1

    Xall = np.asarray(Xall, dtype=np.float32)
    Yall = np.asarray(Yall, dtype=np.int64)

    Xtrain, Ytrain = Xall[:train_size], Yall[:train_size]
    Xtest, Ytest = Xall[train_size:], Yall[train_size:]

    if shuffle_data:
        Xtrain, Ytrain = shuffle(Xtrain, Ytrain)
        Xtest, Ytest = shuffle(Xtest, Ytest)

    return (torch.from_numpy(Xtrain).float(), torch.from_numpy(Ytrain).long()), \
           (torch.from_numpy(Xtest).float(), torch.from_numpy(Ytest).long())


# ----------------------------
# Helpers
# ----------------------------
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def score(y, t):
    return torch.mean((torch.argmax(y, axis=1) == t).double()).item()

def alpha_from_c(c: Tensor, eps: float = 1e-12) -> Tensor:
    """
    α = sqrt(n) / (2 * ||c||) per neuron (batch-safe); grads flow to c.
    (Eqs. 18, 29)
    """
    n = c.shape[-1]
    return (n ** 0.5) / (2.0 * c.norm(dim=-1).clamp_min(eps))  # (K,)

def pairwise_sq_dists(X: Tensor) -> Tensor:
    """
    X: (B, P, n). Return D2: (B, P, P) with ||x_i - x_j||^2.
    """
    XX = (X**2).sum(dim=-1, keepdim=True)   # (B,P,1)
    D2 = XX + XX.transpose(1, 2) - 2.0 * (X @ X.transpose(1, 2))
    return D2.clamp_min(0.0)  # numerical safety

# ----------------------------
# EN-invariant bank + head
# ----------------------------
class ENInvariantBank(nn.Module):
    """
    A bank of K E(n)-invariant neurons implemented in closed form:
        I_ij^k = γ_k * α_k * (2 r_k^2 - 2 ||c_k||^2 - ||x_i - x_j||^2),
    mean-pooled over i<j.  (See Eq. (43) in the draft)
    """
    def __init__(self, n: int, K: int):
        super().__init__()
        self.n, self.K = n, K
        # S = [ c (n), s_{n+1} (1), γ (1) ]
        S0 = torch.cat((0.01*torch.randn(K, n + 1), torch.ones(K, 1)), dim=1)
        self.S = nn.Parameter(S0)       # (K, n+2), fully learnable
        self.gamma = self.S[:, -1]      # view into S (learnable)

    def forward(self, X: Tensor) -> Tensor:
        """
        X: (B, P, n). Returns pooled invariants per neuron: (B, K).
        """
        B, P, n = X.shape
        assert n == self.n, f"Expected last dim {self.n}, got {n}"

        # Split S into (c, s_{n+1}, gamma-view)
        c = self.S[:, :n]               # (K,n)
        c2 = (c**2).sum(dim=-1)         # (K,)
        s_np1 = self.S[:, n]            # (K,)
        # r^2 = ||c||^2 - 2*s_{n+1}
        r2 = c2 - 2.0 * s_np1           # (K,)
        alpha = alpha_from_c(c)         # (K,)

        # Pairwise squared distances (E(n)-invariant)
        D2 = pairwise_sq_dists(X)       # (B,P,P)

        # Upper-triangular mask i<j 
        tri_mask = torch.triu(torch.ones(P, P, device=X.device, dtype=torch.bool), diagonal=1)
        D2u = D2[:, tri_mask]           # (B, M) where M = P*(P-1)/2

        # I_ij^k = γ_k * α_k * (2 r_k^2 - 2 ||c_k||^2 - D2_ij), then mean over pairs
        base = 2.0 * r2 - 2.0 * c2      # (K,)
        I_pairs = (self.gamma * alpha).view(1, 1, self.K) * (
            base.view(1, 1, self.K) - D2u.unsqueeze(-1)  # (B, M, K)
        )
        pooled = I_pairs.mean(dim=1)    # (B,K)
        return pooled


class ENet2L(nn.Module):
    def __init__(self, n: int, K1: int = 10, n_classes: int = 7):
        super().__init__()
        self.bank1 = ENInvariantBank(n, K1)
        self.head  = nn.Linear(K1, n_classes)  # single linear head

    def forward(self, X: Tensor) -> Tensor:
        z1 = self.bank1(X)    # invariant, nonlinear features
        return self.head(z1)


# ----------------------------
# Baseline: per-point MLP + mean pooling (perm-invariant, not E(n)-invariant)
# ----------------------------
class MLPPointPool(nn.Module):
    def __init__(self, n_classes: int = 7):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Linear(3, 10), nn.ReLU(inplace=True),
            nn.Linear(10, 8), nn.ReLU(inplace=True),
        )
        self.head = nn.Linear(8, n_classes)

    def forward(self, X: Tensor) -> Tensor:
        # X: (B,P,3)
        h = self.feat(X)                   # (B,P,8)
        pooled = h.mean(dim=1)             # (B,8)
        return self.head(pooled)           # (B,C)

# ----------------------------
# Quick permutation & E(n) tests
# ----------------------------
# Synthetic batch: B objects, each with P points in R^n
n, P, B, C = 3, 4, 8, 7
X = torch.randn(B, P, n, device=DEVICE).double()

# Instantiate models
m1 = MLPPointPool(n_classes=C).to(DEVICE).double()
m2 = ENet2L(n=n, K1=10, n_classes=C).to(DEVICE).double()

print("Params MLPPointPool       :", count_parameters(m1))
print("Params ENet2L             :", count_parameters(m2))

# Base logits
with torch.no_grad():
    logits_mlp     = m1(X)     # (B,C)
    logits_E       = m2(X)     # (B,C)

# --- Permutation invariance test (reorder points) ---
perm = torch.randperm(P, device=DEVICE)
X_perm = X[:, perm, :]
with torch.no_grad():
    logits_mlp_perm     = m1(X_perm)
    logits_E_perm       = m2(X_perm)

err_mlp_perm     = (logits_mlp - logits_mlp_perm).abs().max().item()
err_E_perm       = (logits_E   - logits_E_perm).abs().max().item()
print("\nPermutation invariance |logits - logits(perm)|_∞")
print(f"MLP    : {err_mlp_perm:.3e} (should be ~0)")
print(f"EN     : {err_E_perm:.3e} (should be ~0)")

# --- E(n) invariance test: O(n) + translation ---
# You said you have this util; otherwise replace with any random orthogonal matrix.
R = random_on_matrix(n, reflection_prob=0.5)         # O(n)
R = torch.as_tensor(R, device=DEVICE, dtype=X.dtype) # ensure tensor on device
t = torch.empty(n, device=DEVICE).uniform_(-3.0, 3.0).double()

X_rt = (X @ R.T) + t.view(1, 1, n)

with torch.no_grad():
    logits_mlp_rt     = m1(X_rt)
    logits_E_rt       = m2(X_rt)

err_mlp_En     = (logits_mlp - logits_mlp_rt).abs().max().item()
err_E_En       = (logits_E   - logits_E_rt).abs().max().item()

print("\nE(n) invariance (logits) |logits - logits(R,t)|_∞")
print(f"MLP    : {err_mlp_En:.3e} (not expected to be invariant)")
print(f"EN     : {err_E_En:.3e} (expected ~0; closed-form invariants, Eq. 43)")

Params MLPPointPool       : 191
Params ENet2L             : 127

Permutation invariance |logits - logits(perm)|_∞
MLP    : 0.000e+00 (should be ~0)
EN     : 2.274e-13 (should be ~0)

E(n) invariance (logits) |logits - logits(R,t)|_∞
MLP    : 8.808e-02 (not expected to be invariant)
EN     : 3.411e-13 (expected ~0; closed-form invariants, Eq. 43)


### Create the dataset

In [5]:
# get the data:
Xtrain, Ytrain = get_tetris_data(only_canonical=True)
_, (Xval, Yval) = get_tetris_data(total_size=7000, train_size=0, distortion=0.0)
_, (Xtest, Ytest) = get_tetris_data(total_size=7000, train_size=0, distortion=0.0)

output_dim = len(set(Ytrain.numpy()))
print(Xtrain, Ytrain)
print(Xtest.shape)

Xtrain, Ytrain, Xval, Yval, Xtest, Ytest = Xtrain.to(DEVICE), Ytrain.to(DEVICE), Xval.to(DEVICE), Yval.to(DEVICE), Xtest.to(DEVICE), Ytest.to(DEVICE)

tensor([[[0., 0., 0.],
         [0., 0., 1.],
         [1., 0., 0.],
         [1., 1., 0.]],

        [[0., 0., 0.],
         [1., 0., 0.],
         [0., 1., 0.],
         [1., 1., 0.]],

        [[0., 0., 0.],
         [0., 0., 1.],
         [0., 0., 2.],
         [0., 0., 3.]],

        [[0., 0., 0.],
         [0., 0., 1.],
         [0., 1., 0.],
         [1., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 1.],
         [0., 0., 2.],
         [0., 1., 0.]],

        [[0., 0., 0.],
         [0., 0., 1.],
         [0., 0., 2.],
         [0., 1., 1.]],

        [[0., 0., 0.],
         [1., 0., 0.],
         [1., 1., 0.],
         [2., 1., 0.]]]) tensor([0, 1, 2, 3, 4, 5, 6])
torch.Size([7000, 4, 3])


## Train the models on the main data

#### E(n)-equivariant spherical decision surfaces

In [6]:
# set the seed here:
torch.manual_seed(SEED)

# instantiate the model:
model = ENet2L(n=n, K1=10, n_classes=output_dim).to(DEVICE)

print(model)
print('total number of parameters:', count_parameters(model))
print()

# define the loss and optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 1000
batch_size = len(Xtrain)
n_batches = len(Xtrain) // batch_size


# train the model:
for i in range(epochs): 
    for j in range(n_batches):          
        Xbatch = Xtrain[j*batch_size:(j+1)*batch_size,]
        Ybatch = Ytrain[j*batch_size:(j+1)*batch_size]

        y_pred = model(Xbatch)
        loss = criterion(y_pred, Ybatch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred.detach_()
        acc = score(y_pred, Ybatch)

        y_val_pred = model(Xval)
        y_val_pred.detach_()
        val_loss = criterion(y_val_pred, Yval)
        val_acc = score(y_val_pred, Yval)

        if i % 100 == 0:
            print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

# evaluate on test set:
Ytest_pred = model(Xtest).detach_()
test_acc = score(Ytest_pred, Ytest)

print('test acc:', np.round(test_acc, 5))

ENet2L(
  (bank1): ENInvariantBank()
  (head): Linear(in_features=10, out_features=7, bias=True)
)
total number of parameters: 127

epoch: 0,  batch: 0,  cost: 90.649,  val_cost: 80.290,  acc:  0.143,  val_acc: 0.143
epoch: 100,  batch: 0,  cost: 1.659,  val_cost: 1.654,  acc:  0.429,  val_acc: 0.286
epoch: 200,  batch: 0,  cost: 1.475,  val_cost: 1.474,  acc:  0.429,  val_acc: 0.429
epoch: 300,  batch: 0,  cost: 1.350,  val_cost: 1.349,  acc:  0.429,  val_acc: 0.429
epoch: 400,  batch: 0,  cost: 1.230,  val_cost: 1.229,  acc:  0.429,  val_acc: 0.429
epoch: 500,  batch: 0,  cost: 1.097,  val_cost: 1.095,  acc:  0.429,  val_acc: 0.429
epoch: 600,  batch: 0,  cost: 0.927,  val_cost: 0.925,  acc:  0.714,  val_acc: 0.714
epoch: 700,  batch: 0,  cost: 0.767,  val_cost: 0.767,  acc:  0.714,  val_acc: 0.714
epoch: 800,  batch: 0,  cost: 0.694,  val_cost: 0.687,  acc:  0.857,  val_acc: 0.714
epoch: 900,  batch: 0,  cost: 0.530,  val_cost: 0.528,  acc:  1.000,  val_acc: 1.000
epoch: 999,  batch

In [7]:
# the learned parameters:
S_learned = model.bank1.S.detach()
c = S_learned[:, :n]               
c2 = (c**2).sum(dim=-1)        
s_np1 = S_learned[:, n]            
# r^2 = ||c||^2 - 2*s_{n+1}
r2 = c2 - 2.0 * s_np1 # negative radii^2 can be learned as well
print("centres:\n", c)
print("radii:\n", torch.sqrt(r2.to(torch.complex64)))
print("gammas:\n", S_learned[:,-1])

centres:
 tensor([[ 1.3302e-02,  3.8886e-03,  3.6007e-07],
        [ 2.8437e-02, -3.2195e-02, -2.9138e-02],
        [-3.3269e-02,  3.9858e-02, -3.1873e-02],
        [-2.6112e-02, -2.5094e-02, -2.6391e-02],
        [ 8.3170e-03,  7.6176e-05, -4.0211e-06],
        [-1.5293e-02,  1.9990e-02,  1.5965e-02],
        [ 3.0121e-02,  2.9249e-02,  3.6377e-02],
        [ 3.1542e-02,  2.5538e-02, -2.9382e-02],
        [-2.8690e-02,  2.2860e-02,  9.6258e-03],
        [ 7.6197e-03,  4.7302e-09, -1.2828e-02]])
radii:
 tensor([0.5198+0.0000j, 0.0000+0.0527j, 0.3408+0.0000j, 0.0000+0.2810j,
        0.6762+0.0000j, 0.0000+0.4824j, 0.4446+0.0000j, 0.5310+0.0000j,
        0.0000+0.4991j, 0.0000+0.7143j])
gammas:
 tensor([1.0159, 0.9742, 0.9711, 0.9786, 1.0216, 0.9982, 0.9744, 0.9829, 0.9914,
        1.0156])


#### MLP

In [8]:
# set the seed here:
torch.manual_seed(SEED)

# instantiate the model:
model = MLPPointPool(n_classes=7).to(DEVICE)

print(model)
print('total number of parameters:', count_parameters(model))
print()

# define the loss and optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 1000
batch_size = len(Xtrain)
n_batches = len(Xtrain) // batch_size

# train the model:
for i in range(epochs): 
    for j in range(n_batches):          
        Xbatch = Xtrain[j*batch_size:(j+1)*batch_size,]
        Ybatch = Ytrain[j*batch_size:(j+1)*batch_size]

        y_pred = model(Xbatch)
        loss = criterion(y_pred, Ybatch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred.detach_()
        acc = score(y_pred, Ybatch)

        y_val_pred = model(Xval)
        y_val_pred.detach_()
        val_loss = criterion(y_val_pred, Yval)
        val_acc = score(y_val_pred, Yval)

        if i % 100 == 0:
            print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

# evaluate on test set:
Ytest_pred = model(Xtest).detach_()
test_acc = score(Ytest_pred, Ytest)

print('test acc:', np.round(test_acc, 5))

MLPPointPool(
  (feat): Sequential(
    (0): Linear(in_features=3, out_features=10, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=10, out_features=8, bias=True)
    (3): ReLU(inplace=True)
  )
  (head): Linear(in_features=8, out_features=7, bias=True)
)
total number of parameters: 191

epoch: 0,  batch: 0,  cost: 1.949,  val_cost: 1.960,  acc:  0.143,  val_acc: 0.143
epoch: 100,  batch: 0,  cost: 1.862,  val_cost: 1.969,  acc:  0.286,  val_acc: 0.150
epoch: 200,  batch: 0,  cost: 1.644,  val_cost: 2.171,  acc:  0.286,  val_acc: 0.154
epoch: 300,  batch: 0,  cost: 1.293,  val_cost: 3.175,  acc:  0.571,  val_acc: 0.156
epoch: 400,  batch: 0,  cost: 1.059,  val_cost: 4.785,  acc:  0.714,  val_acc: 0.158
epoch: 500,  batch: 0,  cost: 0.862,  val_cost: 6.855,  acc:  0.714,  val_acc: 0.151
epoch: 600,  batch: 0,  cost: 0.666,  val_cost: 9.402,  acc:  1.000,  val_acc: 0.150
epoch: 700,  batch: 0,  cost: 0.477,  val_cost: 12.383,  acc:  1.000,  val_acc: 0.150
epoch: 800,  

#### MLP + aug

In [9]:
# set the seed here:
torch.manual_seed(SEED)
np.random.seed(SEED)

# instantiate the model:
model = MLPPointPool(n_classes=7).to(DEVICE)

print(model)
print('total number of parameters:', count_parameters(model))
print()

# define the loss and optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 1000
batch_size = len(Xtrain)
n_batches = len(Xtrain) // batch_size

# train the model:
for i in range(epochs): 
    for j in range(n_batches):          
        Xbatch = Xtrain[j*batch_size:(j+1)*batch_size,]
        Ybatch = Ytrain[j*batch_size:(j+1)*batch_size]

        R = random_on_matrix_batch(len(Xbatch), n, reflection_prob=0.5)        # O(n)
        R = torch.as_tensor(R, device=DEVICE, dtype=Xbatch.dtype) # ensure tensor on device
        t = torch.empty(len(Xbatch), 1, n, device=DEVICE).uniform_(-3.0, 3.0)
        
        Xbatch = Xbatch @ R + t

        y_pred = model(Xbatch)
        loss = criterion(y_pred, Ybatch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred.detach_()
        acc = score(y_pred, Ybatch)

        y_val_pred = model(Xval)
        y_val_pred.detach_()
        val_loss = criterion(y_val_pred, Yval)
        val_acc = score(y_val_pred, Yval)

        if i % 100 == 0:
            print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

# evaluate on test set:
Ytest_pred = model(Xtest).detach_()
test_acc = score(Ytest_pred, Ytest)

print('test acc:', np.round(test_acc, 5))

MLPPointPool(
  (feat): Sequential(
    (0): Linear(in_features=3, out_features=10, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=10, out_features=8, bias=True)
    (3): ReLU(inplace=True)
  )
  (head): Linear(in_features=8, out_features=7, bias=True)
)
total number of parameters: 191

epoch: 0,  batch: 0,  cost: 1.974,  val_cost: 1.960,  acc:  0.143,  val_acc: 0.143
epoch: 100,  batch: 0,  cost: 1.940,  val_cost: 1.949,  acc:  0.143,  val_acc: 0.149
epoch: 200,  batch: 0,  cost: 1.973,  val_cost: 1.947,  acc:  0.000,  val_acc: 0.148
epoch: 300,  batch: 0,  cost: 1.949,  val_cost: 1.946,  acc:  0.000,  val_acc: 0.147
epoch: 400,  batch: 0,  cost: 1.925,  val_cost: 1.945,  acc:  0.000,  val_acc: 0.152
epoch: 500,  batch: 0,  cost: 1.954,  val_cost: 1.944,  acc:  0.000,  val_acc: 0.155
epoch: 600,  batch: 0,  cost: 1.966,  val_cost: 1.943,  acc:  0.000,  val_acc: 0.160
epoch: 700,  batch: 0,  cost: 1.928,  val_cost: 1.944,  acc:  0.286,  val_acc: 0.157
epoch: 800,  b

#### MLP + centred input + aug

In [10]:
# set the seed here:
torch.manual_seed(SEED)
np.random.seed(SEED)

# instantiate the model:
model = MLPPointPool(n_classes=7).to(DEVICE)

print(model)
print('total number of parameters:', count_parameters(model))
print()

# define the loss and optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 1000
batch_size = len(Xtrain)
n_batches = len(Xtrain) // batch_size


# train the model:
for i in range(epochs): 
    for j in range(n_batches):          
        Xbatch = Xtrain[j*batch_size:(j+1)*batch_size,]
        Ybatch = Ytrain[j*batch_size:(j+1)*batch_size]

        R = random_on_matrix_batch(len(Xbatch), n, reflection_prob=0.5)        # O(n)
        R = torch.as_tensor(R, device=DEVICE, dtype=Xbatch.dtype) # ensure tensor on device
        
        # centring and O(3)-augmentation
        Xbatch = Xbatch - Xbatch.mean(dim=1, keepdim=True)
        Xbatch = Xbatch @ R
   
        y_pred = model(Xbatch)
        loss = criterion(y_pred, Ybatch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred.detach_()
        acc = score(y_pred, Ybatch)

        # centring:
        Xval = Xval - Xval.mean(dim=1, keepdim=True)
        y_val_pred = model(Xval)
        y_val_pred.detach_()
        val_loss = criterion(y_val_pred, Yval)
        val_acc = score(y_val_pred, Yval)

        if i % 100 == 0:
            print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

print('epoch: %d,  batch: %d,  cost: %.3f,  val_cost: %.3f,  acc:  %.3f,  val_acc: %.3f' % (i, j, loss.item(), val_loss.item(), acc, val_acc))

# evaluate on test set:
# centring:
Xtest = Xtest - Xtest.mean(dim=1, keepdim=True)
Ytest_pred = model(Xtest).detach_()
test_acc = score(Ytest_pred, Ytest)

print('test acc:', np.round(test_acc, 5))

MLPPointPool(
  (feat): Sequential(
    (0): Linear(in_features=3, out_features=10, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=10, out_features=8, bias=True)
    (3): ReLU(inplace=True)
  )
  (head): Linear(in_features=8, out_features=7, bias=True)
)
total number of parameters: 191

epoch: 0,  batch: 0,  cost: 1.953,  val_cost: 1.953,  acc:  0.143,  val_acc: 0.143
epoch: 100,  batch: 0,  cost: 1.941,  val_cost: 1.944,  acc:  0.143,  val_acc: 0.143
epoch: 200,  batch: 0,  cost: 1.944,  val_cost: 1.940,  acc:  0.286,  val_acc: 0.187
epoch: 300,  batch: 0,  cost: 1.933,  val_cost: 1.935,  acc:  0.143,  val_acc: 0.187
epoch: 400,  batch: 0,  cost: 1.930,  val_cost: 1.926,  acc:  0.143,  val_acc: 0.197
epoch: 500,  batch: 0,  cost: 1.889,  val_cost: 1.911,  acc:  0.286,  val_acc: 0.213
epoch: 600,  batch: 0,  cost: 1.924,  val_cost: 1.888,  acc:  0.286,  val_acc: 0.250
epoch: 700,  batch: 0,  cost: 1.905,  val_cost: 1.857,  acc:  0.429,  val_acc: 0.283
epoch: 800,  b